# SC4001 Phase 3, DAVIS DANN re-test under the corrected loss, with lambda ablation

Accelerator must be **GPU (T4 x2 or P100)**. `src/train_dann.py` refuses CPU by design.
Code and data come from the attached dataset `isaacwjk21/sc4001-project`; no internet or installs.

## What this kernel settles

The first GPU run of Phase 3 (kernel `sc4001-phase3-davis-dann`, 2026-11-05) found DANN at
lambda 1.0 **worse** than the baseline on DAVIS `split_cold_target`: test MSE 0.491 (sd 0.004)
against 0.383 (sd 0.010), CI 0.773 against 0.792, with early stopping selecting epoch 5 or 6
against the baseline's 51 to 93.

The cause was not the expected one. A discriminator that memorises its few distinct proteins
would show accuracy climbing toward 1.0. Instead `history.csv` showed `domain_loss` pinned at
0.690, about ln 2, and accuracy at chance for all 20 epochs: the discriminator never learned
anything, so the encoder was receiving an adversarial gradient derived from noise, scaled by a
lambda that ramped from 0.025 to 0.75.

The implementation explains that. Lambda was applied twice, as the gradient reversal coefficient
and again as a weight on the domain BCE, so the discriminator's own gradient was scaled by
lambda and it trained at 2.5% of its nominal rate in the first epoch. Ganin et al. apply lambda
once. `src/train_dann.py` now does the same: the reversal coefficient carries lambda and the
domain loss is unweighted, giving the encoder an adversarial gradient proportional to lambda and
the discriminator its full gradient from the first step.

Those earlier runs are kept at `outputs/phase3-lambda-squared/` in the repository as the evidence
for this diagnosis. Their numbers must not be pooled with anything produced here.

## Grid

DAVIS `split_cold_target`, `--discriminator-input protein`, seeds 4001/4002/4003 at each of
lambda 1.0, 0.5 and 0.1. Nine runs.

Lambda 1.0 writes to the standard output root so it is directly comparable to the archived runs
at the same nominal setting. The ablations write to `outputs/ablations/lambda<value>` because the
output path carries no lambda and the runs would otherwise overwrite one another.

The last cell prints the first and last epoch of every `history.csv`. If `domain_loss` now moves
off 0.693 and accuracy rises above chance, the discriminator is functioning and any remaining
degradation is a property of the problem rather than of the implementation. If it stays at ln 2
even with a full-rate gradient, then the pooled protein representation genuinely does not
separate training proteins from validation and external ones, which is itself the finding.

Outputs survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the DANN entry point before spending quota: it has never run on a GPU.
smoke = subprocess.run(
    [sys.executable, "src/train_dann.py", "--dataset", "davis",
     "--split", "split_cold_target", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("DANN smoke run failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# Each entry is (lambda_max, seed, output_root).
# train_dann.py writes to outputs/phase3/{dataset}/{split}/seed{seed}/, which carries no lambda,
# so each lambda needs its own SC4001_OUTPUT_ROOT or the runs overwrite one another.
# lambda 1.0 keeps the standard root so it is directly comparable to the archived lambda-squared
# runs at the same nominal setting; the ablations go under outputs/ablations/lambda<value>.
BASE = "/kaggle/working/outputs"
GRID = [(strength, seed,
         BASE if strength == 1.0 else BASE + "/ablations/lambda" + str(strength))
        for strength in (1.0, 0.5, 0.1)
        for seed in (4001, 4002, 4003)]

started = time.time()
status = []
for strength, seed, root in GRID:
    t0 = time.time()
    label = "DANN davis split_cold_target seed" + str(seed) + " lambda" + str(strength)
    print()
    print("===== " + label + " =====", flush=True)
    run = subprocess.run(
        [sys.executable, "src/train_dann.py", "--dataset", "davis",
         "--split", "split_cold_target", "--seed", str(seed),
         "--lambda-max", str(strength), "--discriminator-input", "protein", "--resume"],
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=root), check=False,
    )
    elapsed = time.time() - t0
    status.append((label, run.returncode, elapsed))
    print("----- rc=" + str(run.returncode) + " wall=" + format(elapsed / 60, ".1f")
          + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, elapsed in status:
    print("%-56s rc=%d %6.1f min" % (label, rc, elapsed / 60))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")

# The diagnosis this kernel exists to settle: did the discriminator learn anything this time?
# Under the superseded lambda-squared loss, domain_loss sat at ln 2 = 0.693 for every epoch.
print()
print("Domain-branch diagnostic, first and last epoch of each run:")
for path in sorted(Path("/kaggle/working/outputs").glob("**/history.csv")):
    rows = [line.split(",") for line in path.read_text().strip().splitlines()]
    header, body = rows[0], rows[1:]
    if not body:
        continue
    index = {name: position for position, name in enumerate(header)}
    for tag, row in (("first", body[0]), ("last ", body[-1])):
        print("  %-62s %s epoch %-4s domain_loss %-10.6f acc %-8.4f lambda %-10.6f val_mse %.6f"
              % (str(path.parent).replace("/kaggle/working/outputs/", ""), tag,
                 row[index["epoch"]], float(row[index["domain_loss"]]),
                 float(row[index["domain_accuracy"]]), float(row[index["lambda"]]),
                 float(row[index["val_mse"]])))


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
